# TE02_007 - CAN Hardware Abstraction Compatibility Validation

This notebook validates TE02_007: successful hardware abstraction with a 90% compatibility rate across CAN-enabled machine interfaces.

The validation uses CAN-derived ROS 2 topics only: `/statusjoy`, `/joints`, and `/joint_states`. `/encoders` is intentionally excluded because it comes from external hardware attached to the machine and is not part of the CAN abstraction path.

## Validation Method

The hardware abstraction layer is validated by checking whether CAN-derived machine information is exposed through stable ROS 2 interfaces.

The validation checks:

- `/statusjoy`: discrete CAN-derived machine power/engine state.
- `/joints`: continuous CAN-derived machine joint/position feedback.
- `/joint_states`: standard ROS representation generated from `/joints`.

The KPI compatibility rate is computed as:

`compatibility_rate = passed_checks / total_checks * 100`

The KPI passes when the compatibility rate is at least 90%.

## Workflow

1. Start the ROS 2 environment.
2. Run the acquisition cell in this notebook.
3. Operate the machine or play the rosbag containing `/statusjoy`, `/joints`, and `/joint_states`.
4. Run the analysis cells to generate the compatibility matrix and KPI summary.

If using the TE02_001 rosbag, start the acquisition cell first and then run:

```bash
ros2 bag play /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_001/result/WBag/bag_20260707_121849
```

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32MultiArray, UInt8

KPI_DIR = Path('/home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_007')
KPI_DIR.mkdir(parents=True, exist_ok=True)

RAW_STATUSJOY_CSV = KPI_DIR / 'te02_007_raw_statusjoy.csv'
RAW_JOINTS_CSV = KPI_DIR / 'te02_007_raw_joints.csv'
RAW_JOINT_STATES_CSV = KPI_DIR / 'te02_007_raw_joint_states.csv'
COMPATIBILITY_CSV = KPI_DIR / 'te02_007_compatibility_matrix.csv'
SUMMARY_CSV = KPI_DIR / 'te02_007_summary.csv'

STATUSJOY_TOPIC = '/statusjoy'
JOINTS_TOPIC = '/joints'
JOINT_STATES_TOPIC = '/joint_states'

VALID_STATUSJOY_VALUES = {0, 1, 128}
EXPECTED_MIN_JOINT_CHANNELS = 4
EXPECTED_JOINT_NAMES = [
    'base_to_cylinder_joint',
    'cylinder_to_arm1_joint',
    'arm_link1_to_arm2_joint',
    'arm_link2_to_arm3_joint',
    'arm_link3_to_arm4_joint',
    'arm_link4_to_armFork_joint',
]

ACQUISITION_DURATION_S = 180.0
USE_SIM_TIME = False
MIN_TOPIC_RATE_HZ = 5.0
MAX_TOPIC_GAP_S = 1.0
COMPATIBILITY_THRESHOLD_PCT = 90.0
MIN_MOTION_RANGE = 1e-4

print(f'Output directory: {KPI_DIR}')
print('CAN-derived topics:')
print(f'  {STATUSJOY_TOPIC}')
print(f'  {JOINTS_TOPIC}')
print(f'  {JOINT_STATES_TOPIC}')

In [ ]:
class CANAbstractionLogger(Node):
    def __init__(self):
        super().__init__('te02_007_can_abstraction_logger')
        if USE_SIM_TIME:
            self.set_parameters([rclpy.parameter.Parameter('use_sim_time', rclpy.Parameter.Type.BOOL, True)])

        self.statusjoy_rows = []
        self.joints_rows = []
        self.joint_state_rows = []

        self.create_subscription(UInt8, STATUSJOY_TOPIC, self.statusjoy_cb, 10)
        self.create_subscription(Float32MultiArray, JOINTS_TOPIC, self.joints_cb, 10)
        self.create_subscription(JointState, JOINT_STATES_TOPIC, self.joint_state_cb, 10)
        self.create_timer(10.0, self.health_cb)

    def now_ns(self):
        return self.get_clock().now().nanoseconds

    def statusjoy_cb(self, msg):
        now_ns = self.now_ns()
        value = int(msg.data)
        self.statusjoy_rows.append({
            'time_ns': now_ns,
            'time_s': now_ns / 1e9,
            'statusjoy': value,
            'valid_value': value in VALID_STATUSJOY_VALUES,
        })

    def joints_cb(self, msg):
        now_ns = self.now_ns()
        values = [float(v) for v in msg.data]
        row = {'time_ns': now_ns, 'time_s': now_ns / 1e9, 'channel_count': len(values), 'values': values}
        for idx, value in enumerate(values):
            row[f'joints_{idx}'] = value
        self.joints_rows.append(row)

    def joint_state_cb(self, msg):
        now_ns = self.now_ns()
        row = {'time_ns': now_ns, 'time_s': now_ns / 1e9, 'joint_count': len(msg.name)}
        for name, position in zip(msg.name, msg.position):
            row[name] = float(position)
        self.joint_state_rows.append(row)

    def health_cb(self):
        self.get_logger().info(
            f'samples: /statusjoy={len(self.statusjoy_rows)}, '
            f'/joints={len(self.joints_rows)}, /joint_states={len(self.joint_state_rows)}'
        )

In [ ]:
if not rclpy.ok():
    rclpy.init()

logger = CANAbstractionLogger()
deadline = time.monotonic() + ACQUISITION_DURATION_S
print(f'Acquiring CAN abstraction data for {ACQUISITION_DURATION_S:.1f} s.')
print('Operate the machine or play the rosbag now. Press Ctrl+C here to stop earlier.')

try:
    while time.monotonic() < deadline:
        rclpy.spin_once(logger, timeout_sec=0.1)
except KeyboardInterrupt:
    print('Acquisition interrupted by user.')
finally:
    statusjoy_df = pd.DataFrame(logger.statusjoy_rows)
    joints_df = pd.DataFrame(logger.joints_rows)
    joint_states_df = pd.DataFrame(logger.joint_state_rows)
    logger.destroy_node()

statusjoy_df.to_csv(RAW_STATUSJOY_CSV, index=False)
joints_df.to_csv(RAW_JOINTS_CSV, index=False)
joint_states_df.to_csv(RAW_JOINT_STATES_CSV, index=False)

print(f'/statusjoy samples: {len(statusjoy_df)}')
print(f'/joints samples: {len(joints_df)}')
print(f'/joint_states samples: {len(joint_states_df)}')
display(statusjoy_df.head())
display(joints_df.head())
display(joint_states_df.head())

In [ ]:
def topic_stats(df: pd.DataFrame):
    if df.empty:
        return {'samples': 0, 'duration_s': 0.0, 'rate_hz': 0.0, 'max_gap_s': np.nan}
    if len(df) == 1:
        return {'samples': 1, 'duration_s': 0.0, 'rate_hz': 0.0, 'max_gap_s': np.nan}
    times = df['time_s'].sort_values().to_numpy()
    duration_s = float(times[-1] - times[0])
    gaps = np.diff(times)
    return {
        'samples': int(len(df)),
        'duration_s': duration_s,
        'rate_hz': float((len(df) - 1) / duration_s) if duration_s > 0 else 0.0,
        'max_gap_s': float(np.max(gaps)) if len(gaps) else np.nan,
    }

def add_check(rows, component, check, passed, evidence):
    rows.append({
        'component': component,
        'check': check,
        'status': 'PASS' if bool(passed) else 'FAIL',
        'evidence': evidence,
    })

def finite_columns(df, columns):
    if df.empty or not columns:
        return False
    return bool(np.isfinite(df[columns].to_numpy(dtype=float)).all())

In [ ]:
rows = []

status_stats = topic_stats(statusjoy_df)
add_check(rows, '/statusjoy', 'topic_available', len(statusjoy_df) > 0, f'samples={status_stats["samples"]}')
add_check(rows, '/statusjoy', 'valid_can_state_values', (not statusjoy_df.empty and statusjoy_df['valid_value'].all()), f'observed={sorted(statusjoy_df["statusjoy"].dropna().astype(int).unique().tolist()) if not statusjoy_df.empty else []}')
add_check(rows, '/statusjoy', 'minimum_update_rate', status_stats['rate_hz'] >= MIN_TOPIC_RATE_HZ, f'rate_hz={status_stats["rate_hz"]:.2f}')
add_check(rows, '/statusjoy', 'communication_continuity', (np.isfinite(status_stats['max_gap_s']) and status_stats['max_gap_s'] <= MAX_TOPIC_GAP_S), f'max_gap_s={status_stats["max_gap_s"]}')

joints_stats = topic_stats(joints_df)
joint_value_cols = [c for c in joints_df.columns if c.startswith('joints_')] if not joints_df.empty else []
add_check(rows, '/joints', 'topic_available', len(joints_df) > 0, f'samples={joints_stats["samples"]}')
add_check(rows, '/joints', 'minimum_channel_count', (not joints_df.empty and joints_df['channel_count'].min() >= EXPECTED_MIN_JOINT_CHANNELS), f'min_channels={int(joints_df["channel_count"].min()) if not joints_df.empty else 0}')
add_check(rows, '/joints', 'finite_numeric_values', finite_columns(joints_df, joint_value_cols), f'columns={joint_value_cols}')
add_check(rows, '/joints', 'minimum_update_rate', joints_stats['rate_hz'] >= MIN_TOPIC_RATE_HZ, f'rate_hz={joints_stats["rate_hz"]:.2f}')
add_check(rows, '/joints', 'communication_continuity', (np.isfinite(joints_stats['max_gap_s']) and joints_stats['max_gap_s'] <= MAX_TOPIC_GAP_S), f'max_gap_s={joints_stats["max_gap_s"]}')

if joint_value_cols:
    motion_ranges = joints_df[joint_value_cols].max(numeric_only=True) - joints_df[joint_value_cols].min(numeric_only=True)
    moving_channels = motion_ranges[motion_ranges >= MIN_MOTION_RANGE].index.tolist()
    add_check(rows, '/joints', 'can_feedback_changes_detected', len(moving_channels) > 0, f'moving_channels={moving_channels}, ranges={motion_ranges.to_dict()}')
else:
    add_check(rows, '/joints', 'can_feedback_changes_detected', False, 'no joint value columns available')

joint_states_stats = topic_stats(joint_states_df)
present_joint_names = [name for name in EXPECTED_JOINT_NAMES if name in joint_states_df.columns] if not joint_states_df.empty else []
add_check(rows, '/joint_states', 'topic_available', len(joint_states_df) > 0, f'samples={joint_states_stats["samples"]}')
add_check(rows, '/joint_states', 'expected_joint_names_present', len(present_joint_names) == len(EXPECTED_JOINT_NAMES), f'present={present_joint_names}')
add_check(rows, '/joint_states', 'finite_numeric_values', finite_columns(joint_states_df, present_joint_names), f'checked={present_joint_names}')
add_check(rows, '/joint_states', 'minimum_update_rate', joint_states_stats['rate_hz'] >= MIN_TOPIC_RATE_HZ, f'rate_hz={joint_states_stats["rate_hz"]:.2f}')
add_check(rows, '/joint_states', 'communication_continuity', (np.isfinite(joint_states_stats['max_gap_s']) and joint_states_stats['max_gap_s'] <= MAX_TOPIC_GAP_S), f'max_gap_s={joint_states_stats["max_gap_s"]}')

compatibility_df = pd.DataFrame(rows)
compatibility_df.to_csv(COMPATIBILITY_CSV, index=False)
display(compatibility_df)

In [ ]:
passed_checks = int((compatibility_df['status'] == 'PASS').sum())
total_checks = int(len(compatibility_df))
compatibility_rate_pct = 100.0 * passed_checks / total_checks if total_checks else 0.0
overall_status = 'PASS' if compatibility_rate_pct >= COMPATIBILITY_THRESHOLD_PCT else 'FAIL'

summary_rows = [
    {'metric': 'passed_checks', 'value': passed_checks, 'status': 'INFO', 'reason': ''},
    {'metric': 'total_checks', 'value': total_checks, 'status': 'INFO', 'reason': ''},
    {'metric': 'compatibility_rate_pct', 'value': compatibility_rate_pct, 'status': overall_status, 'reason': f'threshold={COMPATIBILITY_THRESHOLD_PCT}%'},
    {'metric': 'TE02_007_overall', 'value': overall_status, 'status': overall_status, 'reason': f'{passed_checks}/{total_checks} checks passed'},
]

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
print(f'Wrote compatibility matrix: {COMPATIBILITY_CSV}')
print(f'Wrote summary: {SUMMARY_CSV}')
print(f'Overall TE02_007 status: {overall_status}')

## Interpretation

Use `te02_007_compatibility_matrix.csv` as the detailed evidence table. Each row is one CAN hardware abstraction compatibility check.

Use `te02_007_summary.csv` as the KPI result. TE02_007 passes when the compatibility rate is at least 90%.

Manual evidence can be added to the report by comparing `/joints` values with the machine internal monitor. This notebook verifies that the CAN-derived data is available, numeric, continuous, and remapped into the standard ROS `/joint_states` abstraction.